# NB-00 — Data Preparation

Builds the split manifests and the four nested ladder rungs. **Every later notebook reads these
manifests.** Run this once, publish the output as a dataset, then never regenerate splits — if
the splits move, every number in the thesis becomes incomparable.

---

### Kaggle settings

| Setting | Value |
|---|---|
| **Accelerator** | **None (CPU)** — do not waste GPU quota here |
| **Internet** | **ON** |
| **Persistence** | Files only |
| **Input to attach** | `bengaliai-speech` (Add Input → Competitions → search "bengaliai speech" → accept rules) |
| **Output** | Create dataset named **`bn-asr-splits`** |

### What it produces
- `manifest_full.csv` — every usable utterance with path, normalised text, duration, speaker
- `rung_{1h,5h,25h,100h}.csv` — **nested** training subsets
- `dev.csv`, `test.csv` — speaker-disjoint
- `attack_eval_25.csv` — the frozen 25 utterances every attack experiment uses

> **Nesting matters.** 1h ⊂ 5h ⊂ 25h ⊂ 100h. If the rungs were independent samples, a difference
> between them could be sampling noise rather than data volume. Nesting removes that explanation.

> ## ⚠️ DO NOT RE-RUN THIS NOTEBOOK
>
> `bn-asr-splits` is already published and every downstream number is keyed to it. Regenerating
> the splits would silently invalidate any result already recorded. The edits below are for the
> record and for anyone reproducing the work from scratch — they do not change the published
> manifests, and re-running to "pick them up" would cost you the comparability you have.

## 1 · Boilerplate

In [ ]:
import os, sys, json, random, time, re, glob, unicodedata
import numpy as np, pandas as pd

SEED = 1337
random.seed(SEED); np.random.seed(SEED)

WORK = "/kaggle/working"
TEMP = "/kaggle/temp"
os.makedirs(f"{TEMP}/scratch", exist_ok=True)

PROV = {"notebook": "NB-00-data-prep", "seed": SEED,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

print("\nAttached inputs:")
for p in sorted(glob.glob("/kaggle/input/*")):
    print("  ", p)

## 2 · Locate the corpus

In [ ]:
# Kaggle mounts DATASETS at /kaggle/input/<name> but COMPETITIONS one level
# deeper at /kaggle/input/competitions/<name>. Search both, plus one extra level.
PATTERNS = ["/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"]
seen, all_dirs = set(), []
for pat in PATTERNS:
    for p in glob.glob(pat):
        if os.path.isdir(p) and p not in seen:
            seen.add(p); all_dirs.append(p)

KEY = re.compile(r"bengali|bangla|bnai|bn[-_]asr", re.I)
CANDIDATE_ROOTS = sorted(d for d in all_dirs if KEY.search(os.path.basename(d)))

if not CANDIDATE_ROOTS:
    print("Directories visible under /kaggle/input:")
    for d in all_dirs[:40]: print("   ", d)
    raise AssertionError(
        "No Bengali corpus found.\n"
        "Add Input -> Competitions -> search 'bengaliai speech' -> Accept rules -> Add.\n"
        "If it IS attached, add its directory name to the KEY regex above."
    )

ROOT = max(CANDIDATE_ROOTS, key=lambda p: (p.count("/"), len(p)))
print("candidate roots:", CANDIDATE_ROOTS)
print("using root:", ROOT)

# ---------------------------------------------------------------------------
# Do NOT use glob("**/*", recursive=True) here. The corpus holds ~1M mp3 files
# and recursive glob walked all of them twice -- 48 of the first run's 67
# minutes went to this alone. A depth-limited scandir does the same job in
# seconds, because we only need to FIND the directories, not enumerate them.
# ---------------------------------------------------------------------------
AUD_EXT = (".mp3", ".wav", ".flac", ".ogg")

def survey(root, max_depth=2):
    csvs, adirs = [], []
    def walk(d, depth):
        try: entries = list(os.scandir(d))
        except OSError: return
        subdirs, n_audio = [], 0
        for e in entries[:20000]:            # cap: a peek is enough per directory
            if e.is_file(follow_symlinks=False):
                nl = e.name.lower()
                if nl.endswith(".csv"): csvs.append(e.path)
                elif nl.endswith(AUD_EXT): n_audio += 1
            elif e.is_dir(follow_symlinks=False):
                subdirs.append(e.path)
        if n_audio: adirs.append(d)
        if depth < max_depth:
            for s in subdirs: walk(s, depth + 1)
    walk(root, 0)
    return sorted(set(csvs)), sorted(set(adirs))

t0 = time.time()
csvs, aud_dirs = survey(ROOT)
print(f"\nsurveyed in {time.time()-t0:.1f}s  (was ~2900s with recursive glob)")

print("\ncsv files found:")
for c in csvs[:10]: print("   ", c, f"({os.path.getsize(c)/1e6:.1f} MB)")
assert csvs, f"No CSV under {ROOT}."

print("\naudio directories:")
for d in aud_dirs[:10]: print("   ", d)
assert aud_dirs, f"No audio found under {ROOT}."


In [ ]:
train_csv = next((c for c in csvs if os.path.basename(c).lower() == "train.csv"), csvs[0])
df_raw = pd.read_csv(train_csv)
print("loaded:", train_csv)
print("shape :", df_raw.shape)
print("columns:", list(df_raw.columns))
df_raw.head()

## 3 · Unicode normalisation — do it once, here

Bengali text has NFC/NFD variants plus zero-width joiner and non-joiner characters that are
visually invisible but break string equality. Normalising once at source means no downstream
notebook has to think about it, and our targeted-success metric stays meaningful.

In [ ]:
ZW = dict.fromkeys(map(ord, "‌‍"), None)

def norm_bn(s):
    if not isinstance(s, str): return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", s).translate(ZW)).strip()

ID_COL   = df_raw.columns[0]
TEXT_COL = "sentence" if "sentence" in df_raw.columns else df_raw.columns[1]
SPLIT_COL = "split" if "split" in df_raw.columns else None

df = df_raw.rename(columns={ID_COL: "utt_id", TEXT_COL: "text"}).copy()
df["utt_id"] = df["utt_id"].astype(str)
df["text_raw"] = df["text"].astype(str)
df["text"] = df["text_raw"].map(norm_bn)

changed_mask = df["text"] != df["text_raw"]
changed = int(changed_mask.sum())
print(f"normalisation changed {changed} / {len(df)} transcripts "
      f"({100*changed/len(df):.2f}%)")

# ---------------------------------------------------------------------------
# Break the 40% down by CAUSE. A reviewer will ask whether this is a real
# Unicode finding or just whitespace tidying, and the answer is the difference
# between a contribution and a footnote.
# ---------------------------------------------------------------------------
samp = df[changed_mask].sample(min(5000, changed), random_state=SEED)
cause = {"whitespace_only": 0, "zwj_zwnj": 0, "nfc_composition": 0}
for raw, out in zip(samp["text_raw"], samp["text"]):
    if re.sub(r"\s+", " ", raw).strip() == out:
        cause["whitespace_only"] += 1
    elif re.sub(r"\s+", " ", raw.translate(ZW)).strip() == out:
        cause["zwj_zwnj"] += 1
    else:
        cause["nfc_composition"] += 1
tot = sum(cause.values())
print("cause breakdown (sampled):")
for k, v in cause.items():
    print(f"   {k:18s} {v:5d}  ({100*v/tot:.1f}%)")
print("-> NFC composition dominating means this is a genuine Bengali Unicode")
print("   finding, not whitespace tidying. Report the breakdown, not just 40%.")
NORM_CAUSE = {k: round(100*v/tot, 1) for k, v in cause.items()}

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("\nusable rows:", len(df))


## 4 · Resolve audio paths and durations

In [ ]:
# Build an id -> path index once. Globbing per-row is fatally slow on ~1M files.
print("indexing audio files (this takes a few minutes on the full corpus)...")
t0 = time.time()
path_index = {}
for d in aud_dirs:
    for f in os.scandir(d):
        if f.name.endswith((".mp3", ".wav", ".flac", ".ogg")):
            path_index[os.path.splitext(f.name)[0]] = f.path
print(f"indexed {len(path_index)} audio files in {time.time()-t0:.0f}s")

df["path"] = df["utt_id"].map(path_index)
missing = df["path"].isna().sum()
print(f"rows with no audio file: {missing}")
df = df.dropna(subset=["path"]).reset_index(drop=True)
print("rows with audio:", len(df))

In [ ]:
# Duration. Reading headers only is far cheaper than decoding.
import soundfile as sf
from concurrent.futures import ThreadPoolExecutor

def dur_of(p):
    try:
        info = sf.info(p)
        return info.frames / info.samplerate
    except Exception:
        try:                       # mp3 often needs a decoder
            import librosa
            return librosa.get_duration(path=p)
        except Exception:
            return np.nan

SAMPLE_FOR_SPEED = None       # set to e.g. 60000 for a fast first pass
work = df if SAMPLE_FOR_SPEED is None else df.sample(SAMPLE_FOR_SPEED, random_state=SEED)

print(f"measuring duration of {len(work)} files...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    work = work.copy()
    work["duration"] = list(ex.map(dur_of, work["path"].tolist()))
print(f"done in {time.time()-t0:.0f}s")

df = work.dropna(subset=["duration"])
df = df[(df["duration"] >= 1.0) & (df["duration"] <= 15.0)].reset_index(drop=True)
print(f"kept {len(df)} utterances between 1s and 15s | total {df['duration'].sum()/3600:.1f} hours")
df["duration"].describe()

## 5 · Speaker-disjoint splits

Speaker overlap between train and test inflates clean WER, and clean WER is the covariate we
regress out in the iso-WER analysis. If it is contaminated, RQ1 is contaminated.

If the corpus has no speaker column we fall back to a deterministic hash of the utterance id and
**say so explicitly in the thesis**, because it is a real limitation.

In [ ]:
import hashlib

# ---------------------------------------------------------------------------
# SPLIT POLICY.
#
# The first run fell back to hashed pseudo-speakers and produced 953,757
# "speakers" for 953,848 utterances -- one per utterance. That is not a
# speaker-disjoint split, it is a random utterance split, and the same real
# speaker can land in both train and test. Since clean WER is the covariate we
# regress out for iso-WER matching (Section 3.2 of the plan), contaminating it
# weakens the central control of RQ1.
#
# Bengali.AI ships no speaker column, but it DOES ship the organisers' own
# held-out `split`. Respect that instead of inventing a random one.
# ---------------------------------------------------------------------------
SPK_COL = next((c for c in ["speaker", "speaker_id", "client_id", "spk"]
                if c in df.columns), None)
HELD_VALUES = {"valid", "validation", "dev", "test", "eval"}

if SPK_COL:
    df["speaker"] = df[SPK_COL].astype(str)
    SPLIT_MODE = "speaker_disjoint"
    print(f"real speaker column '{SPK_COL}': {df['speaker'].nunique()} speakers")
    spk = sorted(df["speaker"].unique()); random.Random(SEED).shuffle(spk)
    n = len(spk)
    test_spk = set(spk[:max(1, int(0.04*n))])
    dev_spk  = set(spk[max(1, int(0.04*n)):max(2, int(0.08*n))])
    df["split"] = np.where(df["speaker"].isin(test_spk), "test",
                  np.where(df["speaker"].isin(dev_spk), "dev", "train"))

elif SPLIT_COL and (set(df[SPLIT_COL].astype(str).str.lower().unique()) & HELD_VALUES):
    SPLIT_MODE = "organiser_holdout"
    low = df[SPLIT_COL].astype(str).str.lower()
    held = df.index[low.isin(HELD_VALUES)]
    print(f"using the organisers' '{SPLIT_COL}' column: "
          f"{len(held)} held-out rows, {len(df)-len(held)} train")
    # Halve the organisers' held-out set into our dev and test, deterministically.
    h = list(held); random.Random(SEED).shuffle(h)
    dev_idx, test_idx = set(h[:len(h)//2]), set(h[len(h)//2:])
    df["split"] = "train"
    df.loc[df.index.isin(dev_idx),  "split"] = "dev"
    df.loc[df.index.isin(test_idx), "split"] = "test"
    df["speaker"] = "unknown"

else:
    SPLIT_MODE = "random_utterance"
    df["speaker"] = df["utt_id"].map(lambda u: hashlib.md5(u.encode()).hexdigest()[:8])
    print("WARNING: no speaker column and no organiser split -> RANDOM split.")
    rnd = np.random.RandomState(SEED).rand(len(df))
    df["split"] = np.where(rnd < 0.04, "test", np.where(rnd < 0.08, "dev", "train"))

print(f"\nSPLIT_MODE = {SPLIT_MODE}")
if SPLIT_MODE != "speaker_disjoint":
    print("""
LIMITATION TO STATE IN THE THESIS
  Speaker identities are not distributed with this corpus, so our splits are
  disjoint at the RECORDING level, not the SPEAKER level. Clean WER may
  therefore be optimistic relative to a truly speaker-held-out evaluation.
  Mitigation: NB-02 additionally reports clean WER on FLEURS-bn, an
  independent corpus with different speakers and recording conditions, as a
  cross-corpus robustness check on the iso-WER covariate.""")

print()
print(df.groupby("split").agg(utts=("utt_id", "size"),
                              hours=("duration", lambda s: round(s.sum()/3600, 1))))


## 6 · The nested ladder rungs

The core experimental apparatus. Same architecture trained four times; only data volume varies.
Stratified by duration so the rungs differ in *size* alone, not in composition.

In [ ]:
RUNGS = [1, 5, 25, 100]          # hours

train = df[df["split"] == "train"].copy()
avail = train["duration"].sum() / 3600
print(f"available training audio: {avail:.1f} h")
RUNGS = [r for r in RUNGS if r <= avail]
print(f"rungs we can build: {RUNGS} h")

# Duration-stratified shuffle: interleave short/medium/long so any prefix is representative.
train["dur_bin"] = pd.qcut(train["duration"], q=4, labels=False, duplicates="drop")
parts = []
for b, g in train.groupby("dur_bin"):
    g = g.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    g["ord"] = np.arange(len(g)) + b / 10.0
    parts.append(g)
ordered = pd.concat(parts).sort_values("ord").reset_index(drop=True)
ordered["cum_h"] = ordered["duration"].cumsum() / 3600

rung_sets = {}
for h in RUNGS:
    sub = ordered[ordered["cum_h"] <= h]
    rung_sets[h] = sub
    print(f"  rung {h:>3}h -> {len(sub):>7} utts | {sub['duration'].sum()/3600:6.2f} h actual")

# Verify nesting: each rung must be a strict subset of the next.
for a, b in zip(RUNGS, RUNGS[1:]):
    assert set(rung_sets[a]["utt_id"]).issubset(set(rung_sets[b]["utt_id"])), f"{a}h not nested in {b}h"
print("\nnesting verified: each rung is a subset of the next")

## 7 · Freeze the 25-utterance attack evaluation set

Chosen once, now, and never changed. Every eps50 figure in the thesis is measured on exactly
these 40 utterances, so results stay comparable across models, rungs and edit types.

Selection is biased toward utterances containing **না**, because those enable the T1 negation-flip
attack that carries RQ2.

In [ ]:
test = df[df["split"] == "test"].copy()
test = test[(test["duration"] >= 2.0) & (test["duration"] <= 8.0)]

has_na = test[test["text"].str.contains("না", regex=False)]
no_na  = test[~test["text"].str.contains("না", regex=False)]
print(f"test utterances containing না: {len(has_na)} / {len(test)}")

# Cut from 40 to 25: NB-04 measured eps50 at ~17.6 min/utterance, so 40 utts
# x 4 rungs x 2 edit types would have cost ~94 GPU-h. 25 still gives an
# adequate median eps50 with bootstrap CIs.
N_ATTACK_EVAL = 25
n_na = min(15, len(has_na))
pick = pd.concat([
    has_na.sample(n_na, random_state=SEED),
    no_na.sample(N_ATTACK_EVAL - n_na, random_state=SEED),
]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

pick["attack_eval_rank"] = np.arange(len(pick))
assert len(pick) == N_ATTACK_EVAL
print(f"\nfrozen attack set: {len(pick)} utterances | "
      f"{pick['text'].str.contains('না', regex=False).sum()} with না | "
      f"mean {pick['duration'].mean():.1f}s")
pick[["utt_id", "duration", "text"]].head(10)

## 8 · Write everything out

In [ ]:
COLS = ["utt_id", "path", "text", "text_raw", "duration", "speaker", "split"]

df[COLS].to_csv(f"{WORK}/manifest_full.csv", index=False)
df[df.split == "dev"][COLS].to_csv(f"{WORK}/dev.csv", index=False)
df[df.split == "test"][COLS].to_csv(f"{WORK}/test.csv", index=False)
pick[COLS + ["attack_eval_rank"]].to_csv(f"{WORK}/attack_eval_25.csv", index=False)
for h, sub in rung_sets.items():
    sub[COLS].to_csv(f"{WORK}/rung_{h}h.csv", index=False)

# Character vocabulary for the CTC head — built here so every rung shares one vocab.
chars = sorted({c for t in df["text"] for c in t})
vocab = {"<pad>": 0, "<s>": 1, "</s>": 2, "<unk>": 3, "|": 4}
for c in chars:
    if c != " ": vocab[c] = len(vocab)
json.dump(vocab, open(f"{WORK}/vocab.json", "w"), ensure_ascii=False, indent=1)
print(f"vocab size: {len(vocab)} ({len(chars)} distinct characters)")

PROV.update(
    root=ROOT, n_utterances=len(df), total_hours=round(df["duration"].sum()/3600, 2),
    rungs=RUNGS, split_mode=SPLIT_MODE, norm_cause=NORM_CAUSE, vocab_size=len(vocab),
    n_speakers=int(df["speaker"].nunique()),
    normalisation_changed_pct=round(100*changed/len(df_raw), 3),
)
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)
KAGGLE_USER = os.environ.get("KAGGLE_USERNAME") or "YOURUSERNAME"
json.dump({"title": "bn-asr-splits", "id": f"{KAGGLE_USER}/bn-asr-splits",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)

print("\nfiles written:")
for f in sorted(os.listdir(WORK)):
    print(f"  {f:28s} {os.path.getsize(f'{WORK}/{f}')/1e6:8.2f} MB")
print(json.dumps(PROV, indent=2, ensure_ascii=False))

---

## Before you move on

| Gate | Requirement |
|---|---|
| Nesting assertion passed | printed in cell 6 |
| Rung hours within ~2% of target | check the table in cell 6 |
| Splits speaker-disjoint | if `speaker_real` is False, note the limitation in the thesis |
| Attack set frozen at 25 | and at least ~15 contain না |

### Then: Output panel → **Create Dataset** → name it `bn-asr-splits`

Every later notebook attaches that dataset. When you re-run this notebook, use **New Version**
on the same dataset rather than creating a second one.

**Next:** NB-01 trains the ladder. Run it four times, once per rung.